# `opera-rs` Sentinel-1 Bursts, DISP-S1 Frames & Missing-Data Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/opera-rs/blob/main/examples/opera_utils_demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/opera-rs.svg)](https://pypi.org/project/opera-rs/)
[![crates.io](https://img.shields.io/crates/v/opera-rs.svg)](https://crates.io/crates/opera-rs)
[![Docs](https://img.shields.io/badge/docs-opera--rs.opengeos.org-teal.svg)](https://opera-rs.opengeos.org)

This notebook demonstrates the pure-Rust [`opera-utils`](https://github.com/opera-adt/opera-utils) workflows integrated directly into **`opera-rs`**, requiring **zero GDAL/Shapely/GeoPandas dependencies** and **no NASA Earthdata login**:

1. **Sentinel-1 burst IDs & acquisition dates:** Normalize burst IDs, extract dates, and group CSLC-S1 / COMPRESSED-CSLC-S1 filenames (`normalize_burst_id`, `get_burst_id`, `get_dates`, `group_by_burst`)
2. **DISP-S1 filename parsing:** Extract frame ID, polarization, reference/secondary dates, and version from `OPERA_L3_DISP-S1` products (`parse_disp_filename`)
3. **Official `opera-adt/burst_db` v0.9.0 frame & burst database:** Query DISP-S1 frame metadata, UTM/WGS 84 bounds, and frame/burst mappings with SHA-256 verified caching (`get_frame_to_burst_mapping`, `get_burst_ids_for_frame`, `get_frame_ids_for_burst`, `get_frame_bbox`)
4. **Spatial frame intersection & GeoPackage export:** Find all DISP-S1 frames intersecting an AOI across 46,986 global frames (`get_intersecting_frames`), plot them with Matplotlib, and export to GeoPackage (`.gpkg`) via `terrio`
5. **Missing-data subset optimization:** Find optimal rectangular `(burst_ids x dates)` CSLC subsets when acquisitions have gaps (`get_missing_data_options`) and visualize the availability matrix
6. **Public NASA CMR DISP-S1 frame search:** Search `OPERA_L3_DISP-S1_V1` granules by frame ID with automatic version deduplication (`disp_search`) and build offline `xarray` `titiler-cmr` TileJSON URLs
7. **Bundled `opera` CLI subcommands:** Run `burst-id`, `frame`, `frames-intersect`, and `missing-data` directly from `python -m opera_rs`

> **Note:** The `opera-utils` integration functions shown here require `opera-rs >= 0.2.0` (or a local build from `main`).

In [ ]:
%pip install -q -U "opera-rs>=0.2.0" matplotlib pandas

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import opera_rs
import pandas as pd

out_dir = Path("opera_utils_output")
out_dir.mkdir(exist_ok=True)
print(f"opera-rs version: {opera_rs.__version__}")

## 1. Sentinel-1 Burst IDs & Acquisition Dates

Parse and normalize Sentinel-1 burst identifiers (such as `T087-185683-IW2` -> `t087_185683_iw2`), extract `YYYY-MM-DD` dates from filenames or URLs, and group CSLC files by burst ID.

In [ ]:
sample_cslcs = [
    "OPERA_L2_CSLC-S1_T087-185683-IW2_20230322T161649Z_20240504T185235Z_S1A_VV_v1.1.h5",
    "OPERA_L2_CSLC-S1_T087-185683-IW2_20230403T161649Z_20240504T190112Z_S1A_VV_v1.1.h5",
    "OPERA_L2_CSLC-S1_T087-185684-IW2_20230322T161652Z_20240504T185240Z_S1A_VV_v1.1.h5",
    "https://datapool.asf.alaska.edu/CSLC/OPERA_L2_COMPRESSED-CSLC-S1_F11114_T087-185683-IW2_20230101T161649Z_20230101T161649Z_20230322T161649Z_20240505T000000Z_VV_v1.0.h5",
]

print("Normalized burst ID:", opera_rs.normalize_burst_id("T087-185683-IW2"))
for path in sample_cslcs:
    print(f"  burst={opera_rs.get_burst_id(path)} | dates={opera_rs.get_dates(path)}")

grouped = opera_rs.group_by_burst(sample_cslcs)
print("\nGrouped counts by burst:")
for bid, files in grouped.items():
    print(f"  {bid}: {len(files)} file(s)")

## 2. Parse `OPERA_L3_DISP-S1` Filenames

Extract frame ID, acquisition mode, polarization, reference/secondary datetimes, version, and generation timestamp from any `OPERA_L3_DISP-S1` filename, HTTPS URL, or S3 URI.

In [ ]:
disp_url = (
    "https://datapool.asf.alaska.edu/DISP/OPERA-S1/"
    "OPERA_L3_DISP-S1_IW_F11114_VV_20160705T140755Z_20230418T140812Z_v1.0_20241219T231545Z.nc"
)
parsed_disp = opera_rs.parse_disp_filename(disp_url)
pd.Series(parsed_disp)

## 3. Query Official `opera-adt/burst_db` v0.9.0

`opera-rs` fetches the official `opera-adt/burst_db` v0.9.0 release archives lazily on first use, verifies their pinned SHA-256 digests, caches them locally, and queries them directly in Rust.

In [ ]:
frame_id = 11114  # Central California DISP-S1 frame
frame_info = opera_rs.get_frame_to_burst_mapping(frame_id)
print(f"Frame {frame_info['frame_id']} | Native EPSG:{frame_info['epsg']} | Land={frame_info['is_land']} | NA={frame_info['is_north_america']}")
print(f"Constituent bursts: {len(frame_info['burst_ids'])} (first 5: {frame_info['burst_ids'][:5]})")

utm_bbox = opera_rs.get_frame_bbox(frame_id, latlon=False)
wgs_bbox = opera_rs.get_frame_bbox(frame_id, latlon=True)
print("Native UTM bbox:", utm_bbox)
print("WGS 84 bbox:    ", wgs_bbox)

# Reverse lookup: which DISP-S1 frame(s) contain the first burst?
first_burst = frame_info["burst_ids"][0]
parent_frames = opera_rs.get_frame_ids_for_burst(first_burst)
print(f"Burst {first_burst} belongs to frame(s): {parent_frames}")

## 4. Spatial Frame Intersection & GeoPackage Export (Zero GDAL)

Find every DISP-S1 frame across all 46,986 global frames that intersects an Area of Interest (Tulare Lake, California), plot the intersecting polygons with Matplotlib, and export them to a GeoPackage (`.gpkg`) via pure-Rust `terrio`.

In [ ]:
tulare_bbox = [-119.95, 35.85, -119.45, 36.25]
frames_fc = opera_rs.get_intersecting_frames(tulare_bbox)
print(f"Found {len(frames_fc['features'])} DISP-S1 frame(s) intersecting Tulare Lake:")

rows = []
for feat in frames_fc["features"]:
    props = feat.get("properties", {})
    fid = feat.get("id", props.get("frame_id"))
    rows.append({
        "frame_id": fid,
        "orbit_pass": props.get("orbit_pass"),
        "geometry_type": feat["geometry"]["type"],
    })
display(pd.DataFrame(rows))

# Export intersecting frames to GeoPackage (.gpkg) with zero GDAL
gpkg_path = out_dir / "tulare_disp_frames.gpkg"
written = opera_rs.write_vector(frames_fc, gpkg_path, layer_name="disp_frames")
print(f"Wrote {written} frame polygons to {gpkg_path} | read-back bbox: {opera_rs.read_aoi_bbox(gpkg_path)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {"ASCENDING": "#1f77b4", "DESCENDING": "#d95f02"}

for feat in frames_fc["features"]:
    props = feat.get("properties", {})
    fid = feat.get("id", props.get("frame_id"))
    orbit = props.get("orbit_pass") or "UNKNOWN"
    color = colors.get(orbit, "#2ca02c")
    geom = feat["geometry"]
    polys = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    for poly in polys:
        ring = poly[0]
        xs = [pt[0] for pt in ring]
        ys = [pt[1] for pt in ring]
        ax.fill(xs, ys, alpha=0.18, color=color)
        ax.plot(xs, ys, color=color, linewidth=1.6)
        cx, cy = sum(xs[:-1]) / (len(xs) - 1), sum(ys[:-1]) / (len(ys) - 1)
        ax.text(cx, cy, f"F{fid}\n({orbit[:3]})", ha="center", va="center", fontsize=9, fontweight="bold", color=color)

west, south, east, north = tulare_bbox
ax.add_patch(
    mpatches.Rectangle(
        (west, south),
        east - west,
        north - south,
        facecolor="none",
        edgecolor="#d62728",
        linewidth=2.2,
        linestyle="--",
        label="Tulare Lake AOI",
    )
)

handles = [
    mpatches.Patch(color=colors["ASCENDING"], alpha=0.4, label="Ascending frame"),
    mpatches.Patch(color=colors["DESCENDING"], alpha=0.4, label="Descending frame"),
    mpatches.Patch(edgecolor="#d62728", facecolor="none", linestyle="--", linewidth=2, label="Tulare Lake AOI"),
]
ax.legend(handles=handles, loc="lower left")
ax.set_title("OPERA DISP-S1 Frames Intersecting Tulare Lake, CA (burst_db v0.9.0)")
ax.set_xlabel("Longitude (deg)")
ax.set_ylabel("Latitude (deg)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Missing-Data Subset Optimization

InSAR time-series processors require a complete rectangular `(burst_ids x dates)` stack without missing bursts on any retained date. `opera_rs.get_missing_data_options()` ranks optimal subsets maximizing total retained bursts (and executes ~37x to 59x faster than Python on full frame stacks).

In [ ]:
# Simulate a 12-burst x 10-date Sentinel-1 CSLC stack with realistic missing bursts
burst_ids = [f"t042_{88900 + i:06d}_iw{(i % 3) + 1}" for i in range(12)]
dates = [
    "20230101", "20230113", "20230125", "20230206", "20230218",
    "20230302", "20230314", "20230326", "20230407", "20230419",
]
missing_cells = {
    (0, 2), (0, 6),       # burst 0 missing 2 dates
    (3, 4),               # burst 3 missing 1 date
    (7, 2), (7, 8),       # burst 7 missing 2 dates
    (10, 4), (10, 8),     # burst 10 missing 2 dates
}

slc_files = [
    f"OPERA_L2_CSLC-S1_{b.upper().replace('_', '-')}_{dt}T140000Z_20240504T185235Z_S1A_VV_v1.1.h5"
    for bi, b in enumerate(burst_ids)
    for di, dt in enumerate(dates)
    if (bi, di) not in missing_cells
]
print(f"Input CSLC stack: {len(slc_files)} files across {len(burst_ids)} bursts x {len(dates)} dates ({len(missing_cells)} missing)")

options = opera_rs.get_missing_data_options(slc_files)
options_df = pd.DataFrame(
    [
        {
            "rank": idx + 1,
            "num_dates": opt["num_dates"],
            "num_burst_ids": opt["num_burst_ids"],
            "total_num_bursts": opt["total_num_bursts"],
            "excluded_bursts": opt["excluded_bursts"],
            "selected_files": len(opt["inputs"]),
        }
        for idx, opt in enumerate(options[:8])
    ]
)
options_df

In [ ]:
# Visualize the burst x date availability matrix and highlight the #1 optimal subset
best = options[0]
selected_bursts = set(best["burst_ids"])
selected_dates = {d.replace("-", "") for d in best["dates"]}

# Grid codes: 0 = missing, 1 = available (excluded by option #1), 2 = selected by option #1
grid = []
for bi, b in enumerate(burst_ids):
    row = []
    for di, dt in enumerate(dates):
        if (bi, di) in missing_cells:
            row.append(0)
        elif b in selected_bursts and dt in selected_dates:
            row.append(2)
        else:
            row.append(1)
    grid.append(row)

from matplotlib.colors import ListedColormap

cmap = ListedColormap(["#d62728", "#c7c7c7", "#2a9d8f"])
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(grid, cmap=cmap, aspect="auto", vmin=0, vmax=2)
ax.set_xticks(range(len(dates)))
ax.set_xticklabels([f"{d[:4]}-{d[4:6]}-{d[6:]}" for d in dates], rotation=40, ha="right")
ax.set_yticks(range(len(burst_ids)))
ax.set_yticklabels(burst_ids, fontsize=8)
ax.set_title(
    f"Optimal CSLC Subset (#1): {best['num_burst_ids']} bursts x {best['num_dates']} dates = "
    f"{best['total_num_bursts']} bursts retained"
)
legend_patches = [
    mpatches.Patch(color="#2a9d8f", label="Selected (Option #1)"),
    mpatches.Patch(color="#c7c7c7", label="Available (excluded)"),
    mpatches.Patch(color="#d62728", label="Missing acquisition"),
]
ax.legend(handles=legend_patches, loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=3)
plt.tight_layout()
plt.show()

## 6. Public NASA CMR DISP-S1 Frame Search & Version Deduplication

Search `OPERA_L3_DISP-S1_V1` granules directly from NASA's public CMR API by frame ID (or AOI), automatically deduplicating reprocessed product versions by `(frame_id, reference_datetime, secondary_datetime)`.

In [ ]:
disp_res = opera_rs.disp_search(frame_id=11114, limit=25)
print(
    f"Frame 11114 DISP-S1 search: {len(disp_res['products'])} deduplicated product(s) "
    f"(CMR total_hits={disp_res['total_hits']}, duplicates_removed={disp_res['duplicates_removed']})"
)

disp_df = pd.DataFrame(
    [
        {
            "frame_id": p["frame_id"],
            "polarization": p["polarization"],
            "reference_date": p["reference_datetime"][:10],
            "secondary_date": p["secondary_datetime"][:10],
            "version": p["version"],
            "generation_date": p["generation_datetime"][:10],
            "url": p["filename"],
        }
        for p in disp_res["products"][:10]
    ]
)
disp_df

In [ ]:
# Build offline titiler-cmr xarray TileJSON URLs for DISP-S1 displacement visualization
urls = opera_rs.tile_urls(
    "C3294057315-ASF",
    product="disp-s1",
    backend="xarray",
    variables=["displacement"],
    group="/",
)
print("DISP-S1 xarray TileJSON URL:", urls["tilejson_url"])

## 7. Run the Bundled `opera` CLI Subcommands

The `opera-rs` Python wheel bundles the native `opera` CLI and `opera mcp` server, accessible via `opera` or `python -m opera_rs`.

In [ ]:
def run_cli(*args: str) -> str:
    proc = subprocess.run(
        [sys.executable, "-m", "opera_rs", *args],
        check=True,
        capture_output=True,
        text=True,
    )
    return proc.stdout

print("=== opera burst-id ===")
print(run_cli("burst-id", sample_cslcs[0], "--dates", "--json"))

print("=== opera frame 11114 --latlon ===")
frame_json = json.loads(run_cli("frame", "11114", "--latlon", "--json"))
print(f"Frame {frame_json['frame_id']} | native EPSG:{frame_json['epsg']} | bbox EPSG:{frame_json['bbox_epsg']} | bbox={frame_json['bbox']}")

print("\n=== opera frames-intersect (Tulare Lake) ===")
print(run_cli("frames-intersect", "--bbox", "-119.95,35.85,-119.45,36.25", "--ids-only", "--json"))